# Automatic Cointegrated Pair Selection

This notebook automatically finds cointegrated stock pairs from the NIFTY 50 universe.

Steps performed:
1. Load cleaned price dataset
2. Perform pairwise cointegration tests
3. Identify statistically significant pairs
4. Rank pairs based on p-values
5. Save top pairs for further trading analysis


## Import Required Libraries


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import coint

## Load Processed Price Data


In [ ]:
# Load cleaned NIFTY 50 price dataset
close_prices = pd.read_csv(
    "../data/processed/nifty50_close_prices.csv",
    index_col="Date",
    parse_dates=True
)

close_prices.head()

## Define Cointegration Detection Function


In [ ]:
# Function to compute cointegration test for all stock pairs

def find_cointegrated_pairs(data):
    # numbers of assests
    n = data.shape[1]

    # matrices to store test results
    score_matrix = np.zeros((n, n))
    pvalue_matrix = np.ones((n, n))
    
    # list of tickers
    keys = data.columns
    pairs = []

    for i in range(n):
        for j in range(i+1, n):

            S1 = data[keys[i]]
            S2 = data[keys[j]]
            
            # perform Engle-Granger cointegration test
            score, pvalue, _ = coint(S1, S2)

            score_matrix[i, j] = score
            pvalue_matrix[i, j] = pvalue
            
            # store pairs with strong cointegration
            if pvalue < 0.01:
                pairs.append((keys[i], keys[j],pvalue))
    pairs=sorted(pairs,key=lambda x:x[2])  
    pairs=[(a, b) for a ,b, _ in pairs[:20]]          

    return score_matrix, pvalue_matrix, pairs

## Identify Cointegrated Pairs


In [ ]:
scores, pvalues, pairs = find_cointegrated_pairs(close_prices)

print("Total cointegrated pairs found:", len(pairs))
pairs_df = pd.DataFrame(pairs, columns=["Stock1", "Stock2"])


pairs_df.head(20)
print("Top Cointegrated Pairs:")
print(pairs_df.head(10))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,10))

sns.heatmap(
    pvalues,
    xticklabels=close_prices.columns,
    yticklabels=close_prices.columns,
    cmap="coolwarm",
    mask=(pvalues >= 0.05)
)

plt.title("Cointegration P-Value Heatmap")

plt.savefig("../reports/cointegration_heatmap.png")

plt.show()

### Cointegration Results

The above output lists stock pairs whose price series show strong cointegration.

These pairs are potential candidates for statistical arbitrage strategies because their spreads tend to revert to a long-term equilibrium.


In [ ]:
pairs_df = pd.DataFrame(pairs, columns=["Stock1","Stock2"])

pairs_df.to_csv("../data/processed/auto_cointegrated_pairs.csv", index=False)

print("Auto-selected cointegrated pairs saved successfully")


In [ ]:
pvalue_df = pd.DataFrame(
    pvalues,
    index=close_prices.columns,
    columns=close_prices.columns
)

pvalue_df.to_csv("../data/processed/cointegration_pvalues.csv")

print("Cointegration p-value matrix saved")

### Output

The automatically selected cointegrated pairs are saved to:

data/processed/auto_cointegrated_pairs.csv

This dataset will be used in the next notebook to perform multi-pair portfolio backtesting.
